# Sesión 06 - Lab 2: Deduplicación, agregaciones y tuning de rendimiento

Este laboratorio retoma `dbassociate.bronze.interacciones_soporte`: no la tabla `_bruto` del Lab 1, sino Bronze de nuevo, porque la deduplicación conviene resolverla al nivel del ticket completo, antes de aplanar los mensajes, no después. Antes de tocar los datos, arranca con dos configuraciones de paralelismo/memoria que rara vez se ven en un notebook, pero sí aparecen en el examen. Después, deduplicación: dos tickets llegaron reingresados de forma idéntica (`TCK-3005`, `TCK-3011`); otros tres llegaron con una versión actualizada, con distinto estado, distinta última actualización y, en algunos casos, un mensaje nuevo (`TCK-3008`, `TCK-3015`, `TCK-3020`). El plan completo: inspeccionar configuración, deduplicar a nivel de ticket, aplanar con `explode()`, calcular agregaciones de negocio, y cerrar ajustando `shuffle.partitions` y `autoBroadcastJoinThreshold`, verificando cada ajuste directo en el plan de ejecución (`.explain()`).

## Verificación del entorno

In [0]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_06")

print("Tickets en Bronze (con duplicados):")
spark.sql("""
    SELECT COUNT(*) AS filas, COUNT(DISTINCT ticket_id) AS tickets_unicos
    FROM dbassociate.bronze.interacciones_soporte
""").show()

## Lab 2A: spark.default.parallelism vs. spark.sql.shuffle.partitions

Dos configuraciones que se confunden fácil porque ambas hablan de "particiones", pero gobiernan casos distintos. `spark.default.parallelism` es el número de particiones que usa Spark para operaciones sobre RDD, derivado de los cores disponibles en el cluster — un concepto de la API clásica de RDD, no de DataFrame/SQL. `spark.sql.shuffle.partitions` (el que se ajusta más adelante, en el Lab 2G) gobierna específicamente cuántas particiones resultan de un shuffle en una operación de DataFrame/SQL: un `groupBy`, un join que no se resuelve por broadcast. Confundir uno con otro es un error común: cambiar `spark.default.parallelism` no tiene ningún efecto sobre el paralelismo de un `groupBy` en un DataFrame.

Este notebook corre sobre compute serverless con Unity Catalog, que usa Spark Connect: por diseño no expone `SparkContext` (`sc`) ni la API de RDD, así que `spark.default.parallelism` no se puede leer ni ejercitar en vivo desde acá — es un concepto que aparece en el examen, no algo para demostrar con código en este entorno. Lo que sí es ejecutable es confirmar, en el plan de la consulta, que un shuffle real usa exactamente el valor configurado en `shuffle.partitions` — pero no sobre `interacciones_soporte` directamente: al ser una tabla Bronze tan chica, cabe en una sola partición de entrada, y con una sola partición Spark no necesita redistribuir nada para agrupar por `canal`, así que ni siquiera inserta un shuffle. Por eso el `groupBy` de este demo corre sobre un dataset sintético, repartido a propósito en varias particiones, para garantizar que el shuffle sea real y quede visible en el plan.

In [0]:
from pyspark.sql.functions import col, count

df_bronze = spark.table("dbassociate.bronze.interacciones_soporte")

# obtiene la cantida de shuffle.partitions, por defecto son 200
print("spark.sql.shuffle.partitions (configuracion actual):", spark.conf.get("spark.sql.shuffle.partitions"))

df_demo = (
    spark.range(200_000)
    .repartition(8) # coalesce hace todo lo contrario, es decir, reduce el numero de partitions
    .withColumn("canal", (col("id") % 4).cast("string"))
    .groupBy("canal")
    .agg(count("*").alias("total"))
)

# plan de ejecución que ejecutará databricks.
# el driver crea el plan de ejecución
df_demo.explain()

In [0]:
# ejecutamos los transformation
display(df_demo)

**Dónde ver el shuffle:** en el plan, buscar un nodo que incluya la palabra `Exchange` junto a `hashpartitioning(canal#.., N)` — en este compute, con Photon activo, puede aparecer como `PhotonShuffleExchangeSink`/`PhotonShuffleExchangeSource` en vez del `Exchange` genérico de Spark clásico, pero el `hashpartitioning(..., N)` al lado se mantiene igual. Ese `N` es el número de particiones real que Spark va a usar para este `groupBy`, y coincide con `spark.sql.shuffle.partitions`. No hay ningún nodo equivalente para `spark.default.parallelism`: ese concepto no aplica a un DataFrame.

## Lab 2B: Memoria de executor y driver, qué se puede ver desde el notebook

A diferencia de `shuffle.partitions` (ajustable con `spark.conf.set` en cualquier celda), la memoria de executor y driver se configura al crear o editar el cluster, no desde una celda del notebook —y en compute serverless, Databricks la administra automáticamente sin exponerla. Lo que sí es útil para el examen es saber leerla cuando está disponible, para diagnosticar antes de tocar código: un problema de memoria no siempre se resuelve ajustando el DataFrame, a veces el cluster está subdimensionado para el volumen real de datos.

In [0]:
# en serverless estos datos no estan disponibles, porque se asignan dinamicamente
for propiedad in ["spark.executor.memory", "spark.driver.memory"]:
    try:
        print(f"{propiedad}: {spark.conf.get(propiedad)}")
    except Exception:
        print(f"{propiedad}: no expuesto en este tipo de compute (probablemente serverless)")

## Lab 2C: Deduplicar tickets reingresados (mismo patrón de la Sesión 05)

Dos tickets llegaron reingresados de forma idéntica (`TCK-3005`, `TCK-3011`): `dropDuplicates()` sin `subset` alcanza, porque no hay ninguna versión que priorizar. Otros tres llegaron con una versión actualizada (`TCK-3008`, `TCK-3015`, `TCK-3020`): esos necesitan `Window.partitionBy(ticket_id)` + `orderBy(ultima_actualizacion desc)` + `row_number()` + filtro `== 1`, mismo patrón ya practicado en la Sesión 05 (Lab 1D), aplicado acá a una razón de negocio distinta (una actualización de estado, no una reingesta de datos de cliente).

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

filas_antes = df_bronze.count()

df_sin_exactos = df_bronze.dropDuplicates()

ventana_ticket = Window.partitionBy("ticket_id").orderBy(col("ultima_actualizacion").desc())

df_tickets_deduplicados = (
    df_sin_exactos
    .withColumn("version_mas_reciente", row_number().over(ventana_ticket))
    .filter(col("version_mas_reciente") == 1)
    .drop("version_mas_reciente")
)

print("Filas en Bronze (con duplicados):", filas_antes)
print("Filas despues de dropDuplicates() (reingestas exactas fuera):", df_sin_exactos.count())
print("Filas despues de row_number() (una por ticket):", df_tickets_deduplicados.count())

df_tickets_deduplicados.filter(col("ticket_id").isin(["TCK-3008", "TCK-3015", "TCK-3020"])).select(
    "ticket_id", "estado", "prioridad", "ultima_actualizacion"
).show(truncate=False)

## Lab 2D: Aplanar el Bronze ya deduplicado y escribir Silver

Con un ticket = una fila garantizado, este paso combina el `explode()` y la manipulación de columnas del Lab 1 (el `split()` de `categoria_producto`) para escribir de una sola vez la tabla Silver definitiva: ya sin duplicados y con las columnas limpias. A diferencia del Lab 1 (que trabajaba sobre Bronze tal cual), acá el `explode()` ya no arrastra ningún duplicado.

In [0]:
from pyspark.sql.functions import explode, split

df_detalle = (
    df_tickets_deduplicados
    .select(
        "ticket_id", "cliente_id", "canal", "prioridad", "estado", "categoria_producto",
        explode("mensajes").alias("mensaje"),
    )
    .withColumn("categoria", split(col("categoria_producto"), ":").getItem(0))
    .withColumn("codigo_producto", split(col("categoria_producto"), ":").getItem(1))
    .select(
        "ticket_id", "cliente_id", "canal", "prioridad", "estado",
        "categoria", "codigo_producto",
        col("mensaje.autor").alias("autor_mensaje"),
        col("mensaje.texto").alias("mensaje_texto"),
        col("mensaje.timestamp").alias("timestamp_mensaje"),
    )
)

df_detalle.write.mode("overwrite").saveAsTable("dbassociate.silver.interacciones_soporte_detalle")

print("Filas en la tabla Silver final (sin duplicados):", df_detalle.count())

## Lab 2E: Funciones de agregación

`count()` para volumen, `approx_count_distinct()` para una cardinalidad aproximada más barata que un `COUNT DISTINCT` exacto cuando la tabla es grande (acá el ahorro no se nota por el tamaño del dataset, pero el patrón es el mismo en producción), `mean()` para promedio, y `summary()` para un resumen estadístico rápido de una columna numérica.

In [0]:
from pyspark.sql.functions import count, approx_count_distinct, mean

df_mensajes_por_ticket = df_detalle.groupBy("ticket_id").agg(count("*").alias("num_mensajes"))

df_mensajes_por_ticket.select(
    count("*").alias("total_tickets"),
    approx_count_distinct("ticket_id").alias("tickets_unicos_aprox"),
    mean("num_mensajes").alias("promedio_mensajes_por_ticket"),
).show()

# summary es un alias para describe, retorna estadisticas del df (para cada una de sus columnas)
df_mensajes_por_ticket.select("num_mensajes").summary().show()

## Lab 2F: Agregaciones complejas y enriquecimiento con la tabla de SLA

Métricas de negocio por canal y prioridad, enriquecidas con `prioridad_sla` (tabla de referencia chica: cuántas horas tiene cada prioridad para resolverse). `broadcast()` explícito porque la tabla de referencia es minúscula frente a la tabla de tickets, mismo patrón que la Sesión 05.

In [0]:
from pyspark.sql.functions import broadcast, countDistinct

df_sla = spark.read.csv(
    "/Volumes/dbassociate/default/vol_landing/sesion_06/prioridad_sla.csv",
    header=True,
    inferSchema=True,
)
df_sla.write.mode("overwrite").saveAsTable("dbassociate.silver.prioridad_sla")


# broadcast por defecto se aplica a las tablas que tienen menos de 10MB
# si queremos que se aplique a tablas mas grandes, podemos usar broadcast() directamente sobre la tabla.
# si se aplica a un df que pesa GBs o TBs, tendremos problemas de overhead y de memoria
df_metricas = (
    df_tickets_deduplicados
    .groupBy("canal", "prioridad")
    .agg(countDistinct("ticket_id").alias("total_tickets"))
    .join(broadcast(df_sla), on="prioridad", how="left")
)

df_metricas.orderBy(col("total_tickets").desc()).show(truncate=False)

df_metricas.write.mode("overwrite").saveAsTable("dbassociate.silver.metricas_tickets_canal")

## Lab 2G: Verificar shuffle.partitions y autoBroadcastJoinThreshold en el plan de ejecución

Ajustar `spark.sql.shuffle.partitions` o confiar en que Spark elija bien `autoBroadcastJoinThreshold` no sirve de mucho si no se puede confirmar qué pasó realmente. Este lab lo verifica directo en el plan de consulta (`.explain()`), la misma técnica que funciona en cualquier compute, sin depender de mediciones de tiempo sensibles al tamaño del cluster o al momento del día: contra un `groupBy` sintético, el operador `Exchange` del plan muestra el número real de particiones del shuffle (`hashpartitioning(..., N)`), que siempre coincide con `shuffle.partitions`; contra un join con una tabla de referencia que crece, el plan cambia solo entre `BroadcastHashJoin` y `SortMergeJoin` según el tamaño real de esa tabla frente al umbral de `autoBroadcastJoinThreshold` (10MB por default), sin tocarlo a mano.

In [0]:
shuffle_partitions_original = spark.conf.get("spark.sql.shuffle.partitions")

df_sintetico = (
    spark.range(500_000)
    .repartition(8)
    .withColumn("categoria", (col("id") % 20).cast("string"))
)

spark.conf.set("spark.sql.shuffle.partitions", 200) # valor por defecto
df_agrupado_200 = df_sintetico.groupBy("categoria").count()
df_agrupado_200.collect() # collect para forzar en una variable el resultado
print("shuffle.partitions=200 -> plan del groupBy:")
df_agrupado_200.explain()

spark.conf.set("spark.sql.shuffle.partitions", 8) # cambiamos a 8
df_agrupado_8 = df_sintetico.groupBy("categoria").count()
df_agrupado_8.collect()
print("\nshuffle.partitions=8 -> plan del groupBy:")
df_agrupado_8.explain() # el hashPartitioning tiene 8 buckets, porque shuffle.partitions=8

spark.conf.set("spark.sql.shuffle.partitions", shuffle_partitions_original)

**Qué buscar en el plan:** un nodo de shuffle con `hashpartitioning(categoria#.., N)` al lado — en Spark clásico aparece como `Exchange`, en este compute con Photon activo puede aparecer como `PhotonShuffleExchangeSink`/`PhotonShuffleExchangeSource`. Ese `N` es el número real de particiones del shuffle, y cambia de 200 a 8 entre ambas corridas sin tocar nada más del código. Es la misma configuración que gobierna el tiempo de un `groupBy` a gran escala (ver el Lab Reto 2 de esta sesión, con datos reales de cientos de miles de filas); acá se confirma en el plan, no midiendo segundos.

Cada valor es un balance, no una mejora incondicional: `200` particiones reparten mejor el trabajo entre más cores cuando el volumen es grande, pero sobre un shuffle chico generan muchas tareas con poco dato cada una, y el overhead de programar y coordinar esas tareas puede pesar más que el cómputo real. `8` particiones eliminan ese overhead y concentran el trabajo en menos tareas, ideal para shuffles chicos o clusters con pocos cores, pero a mayor volumen cada partición crece más de lo que un executor puede sostener en memoria, forzando derrame a disco (spill) y dejando cores del cluster sin usar por falta de tareas paralelas.

In [0]:
# este pequeño printn no funciona en serverless
print("spark.sql.autoBroadcastJoinThreshold (bytes):", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

df_hechos_sintetico = (
    spark.range(2_000_000)
    .withColumn("prioridad", (col("id") % 4).cast("string"))
)

# realizamos pruebas con 10, 100k y 200k registros
# con 10 aplica el Broadcast, con 100k también pero con 200k ya no porque supero los 10 MB que spark usa por defecto para el Broadcast.
for filas_ref in [10, 100_000, 2_000_000]:
    df_ref_sintetica = (
        spark.range(filas_ref)
        .withColumn("prioridad", (col("id") % 4).cast("string"))
    )
    print(f"\nTabla de referencia con {filas_ref:,} filas:")
    df_hechos_sintetico.join(df_ref_sintetica, on="prioridad", how="left").explain()

**Qué buscar en el plan:** con 10 y 100.000 filas, la tabla de referencia pesa menos que el umbral default (10MB) y el plan muestra un join de tipo broadcast (`BroadcastHashJoin`, o su variante Photon) con un nodo `BroadcastExchange` al lado. Con 2.000.000 de filas ya no entra, y el plan cambia solo a `SortMergeJoin`, con un nodo de shuffle (`Exchange`/`PhotonShuffleExchange...`) a cada lado del join — sin que nadie haya tocado `autoBroadcastJoinThreshold`, es Spark decidiendo con el tamaño real de los datos.

La elección tiene el mismo tipo de costo-beneficio: `BroadcastHashJoin` evita por completo el shuffle de la tabla grande, solo distribuye la tabla chica una vez a cada executor, pero esa tabla chica debe caber íntegra en la memoria de cada executor — forzar un broadcast sobre una tabla que en realidad es grande puede tirar el job por out-of-memory, que es justamente por qué el umbral default es conservador (10MB). `SortMergeJoin` no tiene ese riesgo de memoria porque procesa por lotes ordenados, pero paga el costo de un shuffle a cada lado del join (más tráfico de red y más I/O) incluso cuando una de las tablas era chica y un broadcast hubiera sido más rápido.

## Consulta de validación

In [0]:
spark.sql("""
    SELECT canal, prioridad, total_tickets, sla_horas, departamento_responsable
    FROM dbassociate.silver.metricas_tickets_canal
    ORDER BY total_tickets DESC
""").show(truncate=False)

## Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.interacciones_soporte_detalle")
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.prioridad_sla")
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.metricas_tickets_canal")

print("Tablas temporales de este laboratorio eliminadas.")